In [52]:
import pandas as pd

# Load your data
df1 = pd.read_json("hf://datasets/NxtGenIntern/IT_Job_Roles_Skills_Certifications_Dataset/Top_207_IT_Job_Roles_Skills_Database.json")

# Step 1: Add backend_developer
backend_row = {
    "Job Title": "backend developer",
    "Job Description": (
        "Responsible for server-side web application logic and integration of "
        "the work front-end developers do. Designs, builds, and maintains APIs, "
        "databases, and application infrastructure to ensure high performance, "
        "security, and scalability."
    ),
    "Skills": ", ".join([
        "python",
        "java",
        "c++",
        "sql",
        "nosql",
        "rest apis",
        "docker",
        "kubernetes"
    ]),
    "Certifications": ", ".join([
        "AWS Certified Developer",
        "Oracle Java Certification",
        "Docker Certified Associate",
        "Kubernetes Application Developer"
    ])
}

df1 = pd.concat([df1, pd.DataFrame([backend_row])], ignore_index=True)

# Step 2: Lowercase job titles
df1['Job Title'] = df1['Job Title'].str.lower()

# Step 3: Map original names to model names
job_name_map = {
    "python developer": "python_developer",
    "java developer": "java_developer",
    "web developer": "web_developer",
    "database administrator": "database_administrator",
    "security specialist": "security_analyst",
    "systems administrator": "systems_administrator",
    "agile project manager": "project_manager",
    "front end developer": "frontend_developer",
    "network and systems administrator": "network_administrator",
    "software developer": "software_developer",
    "full stack developer": "fullstack_developer",
    "data scientist": "data_scientist",
    "mobile app developer": "mobile_app_developer",
    "cloud engineer": "cloud_engineer",
    "backend developer": "backend_developer",
    "machine learning engineer": "ml_engineer"
}

df1['Job Title'] = df1['Job Title'].map(job_name_map)

# Step 4: Keep only rows where the job is in model_jobs
model_jobs = list(job_name_map.values())
df1 = df1[df1['Job Title'].isin(model_jobs)].copy()
df1['Skills'] = df1['Skills'].str.lower()
df1.head()


,Job Title,Job Description,Skills,Certifications
24,ml_engineer,Builds machine learning models by implementing...,"python, machine learning, tensorflow, keras, p...",Microsoft Certified: Azure Data Scientist Asso...
40,project_manager,Manages Agile projects by facilitating Scrum c...,"agile methodologies, scrum, kanban, project ma...","Project Management Professional (PMP), Certifi..."
58,cloud_engineer,Builds and maintains cloud infrastructure. Res...,"cloud computing, aws, azure, google cloud, lin...","Google Cloud Associate Cloud Engineering, Comp..."
75,data_scientist,Utilizes statistical analysis and machine lear...,"python, r, machine learning, data science, sta...","Certified Analytics Professional (CAP), Micros..."
76,database_administrator,Responsible for managing database systems incl...,"sql, database management, mysql, postgresql, o...","Oracle Database SQL Certified Associate, Micro..."


In [53]:
skills_from_jobs = (
    df1['Skills']
    .str.split(',')
    .explode()
    .str.strip()
    .str.lower()
    .unique()
)

skills_from_jobs = set(skills_from_jobs)
df2 = pd.read_excel("../../ml/data/skills/skills_enriched.xlsx")
df2 = df2.drop(columns=['skill'])
df2 = df2.rename(columns={'skill_clean': 'skill'})
df2['skill'] = df2['skill'].str.strip().str.lower()
df2 = df2[df2['skill'].isin(skills_from_jobs)].copy()
skill_group_map = {
    "Software Engineering & Development": 1,
    "Web, Mobile & UI/UX Development": 2,
    "IT Operations, Networking & Systems": 3,
    "Cloud Computing & Platforms": 4,
    "Machine Learning & Artificial Intelligence": 5,
    "DevOps, SRE & Platform Engineering": 6,
    "Security, DevSecOps & Compliance": 7,
    "Data Engineering & Big Data": 8,
    "Data Science, Analytics & BI": 9,
    "Infrastructure as Code & Automation": 10
}
df2['skill_group_id'] = df2['skill_group'].map(skill_group_map)
missing_groups = df2[df2['skill_group_id'].isna()]['skill_group'].unique()

if len(missing_groups) > 0:
    raise ValueError(f"Unmapped skill groups found: {missing_groups}")

df2

,skill,skill_group,skill_description,skill_group_id
3,azure,Cloud Computing & Platforms,"Microsoft cloud platform for compute, storage,...",4
4,hadoop,Data Engineering & Big Data,Big data ecosystem for distributed storage and...,8
5,nosql,Software Engineering & Development,"Database technology used to store, query and m...",1
8,security,"Security, DevSecOps & Compliance","Controls and practices to secure systems, mana...",7
9,spark,Data Engineering & Big Data,Distributed processing engine for large-scale ...,8
...,...,...,...,...
382,windows server administration,"IT Operations, Networking & Systems","Operational skills to manage systems, networks...",3
383,linux administration,"IT Operations, Networking & Systems","Operational skills to manage systems, networks...",3
404,angularjs,"Web, Mobile & UI/UX Development",Technologies to build user-facing web/mobile e...,2
407,css3,Cloud Computing & Platforms,"Cloud services and platforms used to deploy, s...",4


In [56]:
# Split, normalize, and explode skills
df1_skills = (
    df1
    .assign(Skills=df1['Skills'].str.split(','))
    .explode('Skills')
)

df1_skills['Skills'] = (
    df1_skills['Skills']
    .str.strip()
    .str.lower()
)

# Drop duplicates just in case
df1_skills = df1_skills.drop_duplicates(
    subset=['Job Title', 'Skills']
)

# Build jobs_dict
jobs_dict = (
    df1_skills
    .groupby('Job Title')['Skills']
    .apply(list)
    .to_dict()
)

import json

with open("jobs_dict.json", "w", encoding="utf-8") as f:
    json.dump(jobs_dict, f, indent=2, ensure_ascii=False)

In [58]:
# Normalize skill names
df2['skill'] = df2['skill'].str.strip().str.lower()

skill_info = {
    row['skill']: {
        'group': row['skill_group'],
        'group_id': row['skill_group_id'],
        'description': row['skill_description']
    }
    for _, row in df2.iterrows()
}

with open("skills_info.json", "w", encoding="utf-8") as f:
    json.dump(skill_info, f, indent=2, ensure_ascii=False)


In [ ]:
df["Skills"] = df["Skills"].apply(lambda x: [s.strip() for s in x.split(",")])
df["Certifications"] = df["Certifications"].apply(lambda x: [c.strip() for c in x.split(",")])

In [ ]:
# Step 1: Create an empty dictionary to hold merged data
merged_data = {}

# Step 2: Loop over each row in the DataFrame
for index, row in df.iterrows():
    job = row["Job Title"]
    skills = row["Skills"]           # should be a list
    certifications = row["Certifications"]  # should be a list
    
    if job not in merged_data:
        merged_data[job] = {
            "Skills": set(skills),
            "Certifications": set(certifications)
        }
    else:
        merged_data[job]["Skills"].update(skills)
        merged_data[job]["Certifications"].update(certifications)

# Step 3: Convert the dictionary back to a DataFrame
df_merged = pd.DataFrame({
    "Job Title": list(merged_data.keys()),
    "Skills": [list(v["Skills"]) for v in merged_data.values()],
    "Certifications": [list(v["Certifications"]) for v in merged_data.values()]
})

df_merged.head()


,Job Title,Skills,Certifications
0,Admin Big Data,"[MapReduce, Data Lakes, Data Modeling, Cloud C...","[AWS Certified Big Data – Specialty, Hortonwor..."
1,Ansible Operations Engineer,"[Infrastructure as Code, Kubernetes, Ansible, ...","[AWS Certified DevOps Engineer – Professional,..."
2,Artifactory Administrator,"[Gradle, Build Automation, Git, CI/CD, Jenkins...","[DevOps Institute Certifications, JFrog Artifa..."
3,Artificial intelligence / Machine Learning Eng...,[],[]
4,Artificial Intelligence / Machine Learning Leader,"[Communication, Data Science, Project Manageme...",[Certified Artificial Intelligence Practitione...


In [8]:
df_merged.describe()

,Job Title,Skills,Certifications
count,200,200,200
unique,200,174,121
top,Admin Big Data,"[SQL, Algorithms, HTML, Data Structures, Git, ...",[]
freq,1,8,50


In [9]:
skill_dict = {}

for index, row in df.iterrows():
    job = row["Job Title"]
    for skill in row["Skills"]:
        skill_lower = skill.lower()
        if skill_lower not in skill_dict:
            skill_dict[skill_lower] = [job]
        else:
            skill_dict[skill_lower].append(job)

print(skill_dict)

{'hadoop': ['Admin Big Data', 'Big Data Architect', 'Big Data Engineer', 'Big Data Specialist', 'Principle Engineer in Big Data', 'Data Engineer', 'DATA SCIENTIST'], 'spark': ['Admin Big Data', 'Big Data Architect', 'Big Data Engineer', 'Big Data Specialist', 'Principle Engineer in Big Data', 'Data Engineer', 'DATA SCIENTIST'], 'mapreduce': ['Admin Big Data', 'Big Data Architect'], 'data lakes': ['Admin Big Data', 'Big Data Architect'], 'data warehousing': ['Admin Big Data', 'Big Data Architect', 'Big Data Engineer', 'Data Architect', 'Principle Engineer in Big Data', 'Data Engineer', 'DATA MODELER'], 'big data architecture': ['Admin Big Data', 'Principle Engineer in Big Data'], 'nosql': ['Admin Big Data', 'Big Data Architect', 'Big Data Engineer', 'Big Data Specialist', 'Data Architect'], 'data modeling': ['Admin Big Data', 'Big Data Architect', 'Big Data Specialist', 'Data Analysts', 'Data Architect', 'Principle Engineer in Data Analysis', 'DATA ANALYST', 'Data Engineer', 'DATA MODEL

In [21]:
with open("alex.txt", "r") as f:
    text = f.read()  # read entire file as one string

words = text.split()
# Normalize to lowercase
words= [s.lower() for s in words]

# Count how many skills match per job
job_match_count = {}

for skill in words:
    if skill in skill_dict:
        for job in skill_dict[skill]:
            job_match_count[job] = job_match_count.get(job, 0) + 1

job_match_list = list(job_match_count.items())
job_match_list.sort(key=lambda item: item[1], reverse=True)
job_match_sorted = job_match_list
for job, count in job_match_sorted:
    print(f"{job}: {count} matching skills")

print(job_match_sorted)


DevOps Engineer: 10 matching skills
Machine Learning Engineer: 10 matching skills
Senior DevOps Engineer: 10 matching skills
Full Stack Developer: 9 matching skills
Full Stack Python Developer/Programmer/Engineer: 9 matching skills
Entry Level Developer: 8 matching skills
Entry Level Software Developer: 8 matching skills
Entry Level Software Engineer: 8 matching skills
Jr Developer: 8 matching skills
Junior Developer: 8 matching skills
Junior Software Developer: 8 matching skills
Junior Software Engineer: 8 matching skills
New Grad Software Engineer: 8 matching skills
Data Analysts: 7 matching skills
DATA ANALYST: 7 matching skills
DATA SCIENTIST: 7 matching skills
Python Developer: 7 matching skills
Big Data Engineer: 6 matching skills
Principle Engineer in Data Analysis: 6 matching skills
Data Engineer: 6 matching skills
FRAMEWORKS SPECIALIST: 6 matching skills
Ansible Operations Engineer: 5 matching skills
Artificial Intelligence Researcher: 5 matching skills
Principle Engineer in A

In [22]:
with open("fiona.txt", "r") as f:
    text = f.read()  # read entire file as one string

words = text.split()
# Normalize to lowercase
words= [s.lower() for s in words]

# Count how many skills match per job
job_match_count = {}

for skill in words:
    if skill in skill_dict:
        for job in skill_dict[skill]:
            job_match_count[job] = job_match_count.get(job, 0) + 1

job_match_list = list(job_match_count.items())
job_match_list.sort(key=lambda item: item[1], reverse=True)
job_match_sorted = job_match_list
for job, count in job_match_sorted:
    print(f"{job}: {count} matching skills")

print(job_match_sorted)


Data Analysts: 4 matching skills
DevOps Engineer: 4 matching skills
Machine Learning Engineer: 4 matching skills
Senior DevOps Engineer: 4 matching skills
DATA ANALYST: 4 matching skills
Big Data Engineer: 3 matching skills
Principle Engineer in Data Analysis: 3 matching skills
Data Engineer: 3 matching skills
DATA SCIENTIST: 3 matching skills
Entry Level Developer: 3 matching skills
Entry Level Software Developer: 3 matching skills
Entry Level Software Engineer: 3 matching skills
Full Stack Developer: 3 matching skills
Full Stack Python Developer/Programmer/Engineer: 3 matching skills
Jr Developer: 3 matching skills
Junior Developer: 3 matching skills
Junior Software Developer: 3 matching skills
Junior Software Engineer: 3 matching skills
New Grad Software Engineer: 3 matching skills
Python Developer: 3 matching skills
WordPress Developer: 2 matching skills
Ansible Operations Engineer: 2 matching skills
Artificial Intelligence Researcher: 2 matching skills
Principle Engineer in Artifi

In [29]:
with open("fiona_eng.txt", "r") as f:
    text = f.read()  # read entire file as one string

words = text.split()
# Normalize to lowercase
words= [s.lower() for s in words]

# Count how many skills match per job
job_match_count = {}

for skill in words:
    if skill in skill_dict:
        for job in skill_dict[skill]:
            job_match_count[job] = job_match_count.get(job, 0) + 1

job_match_list = list(job_match_count.items())
job_match_list.sort(key=lambda item: item[1], reverse=True)
job_match_sorted = job_match_list
for job, count in job_match_sorted:
    print(f"{job}: {count} matching skills")

print(job_match_sorted)


Data Analysts: 4 matching skills
DevOps Engineer: 4 matching skills
Machine Learning Engineer: 4 matching skills
Senior DevOps Engineer: 4 matching skills
DATA ANALYST: 4 matching skills
Search Engine Optimization: 3 matching skills
Artificial Intelligence / Machine Learning Leader: 3 matching skills
Artificial Intelligence / Machine Learning Sr.Leader: 3 matching skills
AGILE PROJECT MANAGER: 3 matching skills
DEVOPS MANAGER: 3 matching skills
Director of Engineering: 3 matching skills
PRODUCT MANAGER: 3 matching skills
Confluence Engineer: 3 matching skills
Big Data Engineer: 3 matching skills
Principle Engineer in Data Analysis: 3 matching skills
Data Engineer: 3 matching skills
DATA SCIENTIST: 3 matching skills
Entry Level Developer: 3 matching skills
Entry Level Software Developer: 3 matching skills
Entry Level Software Engineer: 3 matching skills
Full Stack Developer: 3 matching skills
Full Stack Python Developer/Programmer/Engineer: 3 matching skills
Jr Developer: 3 matching ski